  <img src="https://github.com/gefero/factor_data_tuto_NLP_SICSS/blob/main/imgs/logo_final_conjunto.png?raw=true" width="80%"># Summer Institute in Computational Social Sciences - Buenos Aires 2026# Taller: Procesamiento de Lenguaje Natural y polarización# Desafío: un índice de polarización en comentarios de noticias### Profesor: Germán Rosati

# IntroducciónEn las notebooks anteriores clasificamos **textos sueltos**: cada tuit de HatEval entrabaal modelo, salía con una etiqueta, y evaluábamos qué tan seguido esa etiqueta coincidíacon la de los anotadores humanos. La unidad de análisis era el texto y la pregunta era devalidación: *¿el modelo acierta?*Acá damos un paso que es el que más aparece en la investigación social real y que casinunca se enseña: **la clasificación no es el resultado, es un insumo**. Lo que nosinteresa no es qué sentimiento tiene un comentario en particular, sino una propiedad deun **colectivo** —la sección de comentarios de una noticia— que ningún comentarioindividual tiene por sí solo. Un comentario no es polarizado; una *conversación* lo es.El plan es el siguiente. Tomamos el dataset[`finiteautomata/news-argentina`](https://huggingface.co/datasets/finiteautomata/news-argentina),que reúne comentarios de lectores a noticias de medios argentinos. Clasificamos cadacomentario con dos modelos de [`pysentimiento`](https://github.com/pysentimiento/pysentimiento)—el mismo que usamos en la notebook de discurso de odio, pero con las tareas de**sentimiento** y **emoción** en vez de `hate_speech`—. Y después **agregamos** esasclasificaciones por noticia en un índice de polarización de tres dimensiones:| Dimensión | Qué mide | Insumo ||---|---|---|| **D1** — disenso de valencia | Cuán dividida está la sección entre comentarios positivos y negativos | analyzer `sentiment` || **D2** — violencia emocional | Qué proporción de la emoción expresada es hostil (enojo, asco) y no de otro tipo | analyzer `emotion` || **D3** — bimodalidad | Cuánto se concentra la opinión en los extremos en vez del centro | probabilidades de `sentiment` |Sobre el final vamos a ver que estas tres dimensiones **no miden lo mismo**, y queconstruir el índice obliga a tomar decisiones —qué emociones cuentan como violentas,cuántos comentarios hacen falta para que el índice sea confiable, cómo ponderar lasdimensiones— que no las resuelve el modelo: las resuelve quien investiga. Ese esrealmente el tema de esta notebook.

# Preparación del entornoIgual que en la notebook de discurso de odio, conviene correr esto en Colab **con GPU**(`Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`). Sin GPU lapredicción sobre varios miles de comentarios pasa de un par de minutos a bastante más demedia hora.

In [ ]:
## Instalamos pysentimiento y la librería de datasets de HuggingFace!pip install -q pysentimiento datasets

In [ ]:
# Importamos las librerías necesariasimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport matplotlib.colors as mcolorsimport seaborn as snsfrom datasets import load_datasetfrom sklearn.decomposition import PCAfrom tqdm.auto import tqdmfrom pysentimiento import create_analyzerfrom pysentimiento.preprocessing import preprocess_tweetimport warningswarnings.filterwarnings('ignore')

## ParámetrosTodo lo que conviene tocar está acá arriba. `N_NOTICIAS` controla el costo de la notebook:con 200 noticias y GPU la predicción tarda unos pocos minutos. Si querés correrla sin GPU,bajalo a 30 o 40.`MIN_COMENTARIOS` es más interesante de lo que parece: es el mínimo de comentarios quetiene que tener una noticia para que calculemos su índice. Más abajo justificamos esenúmero en vez de imponerlo.

In [ ]:
SEED = 42                 # semilla, para que la muestra sea reproducibleN_NOTICIAS = 200          # cuántas noticias muestreamosMIN_COMENTARIOS = 30      # mínimo de comentarios para calcular el índice de una noticiaBATCH_SIZE = 64           # tamaño del lote para la predicciónrng = np.random.default_rng(SEED)# Paleta del taller, la misma de las notebooks anterioresFONDO = '#fcfcfb'AZUL = '#2a78d6'ROJO = '#e34948'GRIS = '#898781'AZUL_OSCURO = '#184f95'blue_ramp = mcolors.LinearSegmentedColormap.from_list(    'blue_seq', ['#fcfcfb', '#cde2fb', '#86b6ef', '#3987e5', '#184f95'])def estilo(ax):    """Aplica el estilo visual del taller a un eje."""    ax.set_facecolor(FONDO)    ax.tick_params(colors=GRIS)    for spine in ax.spines.values():        spine.set_color(GRIS)

# Carga de los datos

In [ ]:
ds = load_dataset("finiteautomata/news-argentina")ds

## Qué trae el datasetAntes de escribir una sola línea de análisis, hay que mirar qué hay adentro. Esto queparece un trámite es la parte donde se descubren la mitad de los problemas: columnas queno son lo que su nombre sugiere, comentarios duplicados, noticias sin identificador.

In [ ]:
# Nos quedamos con el primer split disponible y lo pasamos a pandassplit = list(ds.keys())[0]crudo = ds[split].to_pandas()print(f'Split: {split}')print(f'Filas: {len(crudo):,}')print(f'\nColumnas y tipos:')print(crudo.dtypes)crudo.head(3)

## Mapeo de columnasEl resto de la notebook no usa los nombres de columna directamente: usa **roles**(`texto`, `noticia`, `titulo`, `medio`, `fecha`) que se resuelven en la celda de abajo.La ventaja es que si el dataset cambia de esquema —o si querés correr todo esto sobreotro corpus de comentarios— solo tenés que tocar este diccionario y nada más.Los dos roles imprescindibles son `texto` (el comentario) y `noticia` (a qué noticiaresponde). Sin ese segundo no hay índice posible, porque no habría a qué agregar. Losotros tres son opcionales y solo habilitan análisis adicionales.

In [ ]:
# Nombres de columna candidatos para cada rol, en orden de preferenciaCANDIDATOS = {    'texto':   ['text', 'comment', 'body', 'tweet', 'comentario', 'texto', 'content'],    'noticia': ['article_id', 'news_id', 'article', 'note_id', 'id_noticia', 'parent_id'],    'titulo':  ['title', 'headline', 'titulo', 'article_title'],    'medio':   ['outlet', 'medio', 'source', 'newspaper', 'media'],    'fecha':   ['created_at', 'date', 'fecha', 'published_at', 'datetime'],}def detectar_columnas(df, candidatos=CANDIDATOS):    """Mapea cada rol al nombre de columna real, sin distinguir mayúsculas."""    disponibles = {c.lower(): c for c in df.columns}    return {rol: next((disponibles[o] for o in opciones if o in disponibles), None)            for rol, opciones in candidatos.items()}COLS = detectar_columnas(crudo)# Si la detección automática falla, corregí a mano acá. Por ejemplo:# COLS['texto'] = 'nombre_real_de_la_columna'print('Mapeo de columnas detectado:')for rol, col in COLS.items():    print(f'  {rol:<8} -> {col}')faltantes = [r for r in ('texto', 'noticia') if COLS[r] is None]if faltantes:    raise ValueError(        f'No se detectaron las columnas para los roles {faltantes}. '        f'Columnas disponibles: {list(crudo.columns)}. '        f'Corregí el diccionario COLS a mano.'    )

### Si este dataset no tuviera comentariosEl índice necesita una estructura **noticia → muchos comentarios**. Si al inspeccionarel dataset resultara que solo trae los artículos (título y cuerpo) sin las respuestas delos lectores, el reemplazo directo es[`piuba-bigdata/contextualized_hate_speech`](https://huggingface.co/datasets/piuba-bigdata/contextualized_hate_speech),del mismo autor: comentarios de lectores a noticias de Clarín, Infobae, La Nación, Perfily Crónica, con el texto de la noticia como contexto. Basta con cambiar el`load_dataset(...)` de arriba; el mapeo `COLS` hace que todo lo demás siga funcionandosin tocar una línea.

## MuestreoDos filtros. Primero nos quedamos solo con las noticias que tienen al menos`MIN_COMENTARIOS` comentarios, porque un índice calculado sobre cinco comentarios esruido. Después muestreamos `N_NOTICIAS` de esas, para que la notebook corra en un tiemporazonable.Ojo con lo que esto implica: **la muestra no es representativa del universo de noticias**.Al exigir un piso de comentarios nos quedamos, por construcción, con las noticias quegeneraron conversación. Cualquier conclusión del tipo "las noticias argentinas polarizanmucho" está mal: lo que podemos decir es algo sobre las noticias *comentadas*.

In [ ]:
por_noticia = crudo.groupby(COLS['noticia']).size()print(f'Noticias en total: {len(por_noticia):,}')print(f'Comentarios por noticia: mediana={por_noticia.median():.0f}, '      f'media={por_noticia.mean():.1f}, máx={por_noticia.max():,}')elegibles = por_noticia[por_noticia >= MIN_COMENTARIOS].indexprint(f'Noticias con al menos {MIN_COMENTARIOS} comentarios: {len(elegibles):,}')muestra_ids = pd.Series(list(elegibles)).sample(    n=min(N_NOTICIAS, len(elegibles)), random_state=SEED)comentarios = crudo[crudo[COLS['noticia']].isin(set(muestra_ids))].reset_index(drop=True)print(f'\nMuestra final: {comentarios[COLS["noticia"]].nunique():,} noticias, '      f'{len(comentarios):,} comentarios')

In [ ]:
# Cómo se distribuye la cantidad de comentarios por noticiafig, axes = plt.subplots(1, 2, figsize=(11, 3.5))fig.patch.set_facecolor(FONDO)axes[0].hist(por_noticia.values, bins=50, color=AZUL, edgecolor=FONDO)axes[0].axvline(MIN_COMENTARIOS, color=ROJO, linestyle='--',                label=f'MIN_COMENTARIOS = {MIN_COMENTARIOS}')axes[0].set_title('Comentarios por noticia (todo el corpus)')axes[0].set_yscale('log')axes[0].set_ylabel('Noticias (escala log)')axes[0].legend(frameon=False)axes[1].hist(comentarios.groupby(COLS['noticia']).size().values, bins=30,             color=AZUL_OSCURO, edgecolor=FONDO)axes[1].set_title('Comentarios por noticia (muestra)')for ax in axes:    estilo(ax)plt.tight_layout()plt.show()

La distribución es fuertemente asimétrica: muchísimas noticias con pocos comentarios yun puñado con miles. Es la forma típica de la participación online, y es la razón por laque el eje vertical está en escala logarítmica.

## PreprocesamientoVale exactamente lo mismo que dijimos en la notebook de discurso de odio: acá **no**limpiamos agresivamente. Sacar acentos, pasar a minúsculas o reemplazar números tienesentido cuando el objetivo es achicar el vocabulario de un `CountVectorizer`, pero unTransformer preentrenado sobre texto real aprovecha justamente esas señales —mayúsculassostenidas, signos de exclamación repetidos, emojis— que ese tipo de limpieza destruye.Y en comentarios de lectores esas señales son informativas: escribir en mayúsculas *es*parte del contenido emocional.Usamos `preprocess_tweet`, que normaliza el texto igual que se normalizó el corpus con elque se entrenó RoBERTuito: menciones a `@usuario`, URLs a `url`, hashtags separados enpalabras y emojis convertidos a su descripción textual.

In [ ]:
comentarios['text_prep'] = comentarios[COLS['texto']].astype(str).apply(    lambda t: preprocess_tweet(t, lang='es'))comentarios[[COLS['texto'], 'text_prep']].sample(3, random_state=SEED)

Una advertencia técnica que conviene tener presente: los modelos de `pysentimiento`truncan la entrada a 128 tokens. Para comentarios de lectores casi nunca es un problema—la enorme mayoría son mucho más cortos—, pero si en tu corpus hubiera comentarios largos,del texto que excede ese límite el modelo no ve nada.

# Los analyzersCreamos los dos analizadores. La primera vez cada uno descarga su modelo desde HuggingFace(unos cientos de MB); después quedan en caché.

In [ ]:
analyzer_sent = create_analyzer(task="sentiment", lang="es")analyzer_emo = create_analyzer(task="emotion", lang="es")

In [ ]:
ejemplos = [    "Excelente medida, era hora de que alguien se anime a hacer algo así",    "Son todos unos chorros, que se vayan todos de una vez",    "La nota no dice nada nuevo, ya se sabía desde la semana pasada",    "Me da mucha tristeza leer esto, pobre familia",    "Da asco este gobierno de delincuentes, habría que echarlos a patadas",]for texto in ejemplos:    r_sent = analyzer_sent.predict(texto)    r_emo = analyzer_emo.predict(texto)    print(f'Comentario: {texto}')    print(f'  sentimiento: {r_sent.output}  (probas: '          f'{ {k: round(v, 2) for k, v in r_sent.probas.items()} })')    print(f'  emoción:     {r_emo.output}  (probas: '          f'{ {k: round(v, 2) for k, v in sorted(r_emo.probas.items(), key=lambda x: -x[1])[:3]} })')    print()

### Qué devuelve cada analyzerA diferencia de `hate_speech`, que era multilabel, estas dos tareas son de**clasificación simple**: `res.output` es una única etiqueta y `res.probas` un diccionariocon la probabilidad de cada clase.| Tarea | Etiquetas ||---|---|| `sentiment` | `POS`, `NEU`, `NEG` || `emotion` | `others`, `joy`, `sadness`, `surprise`, `disgust`, `fear`, `anger` |El analyzer de emoción está fine-tuneado sobre **EmoEvent**, un corpus de tuits en españolanotados con las seis emociones básicas de Ekman más una categoría `others` para todo loque no encaja. Esa categoría `others` es la mayoritaria por lejos, y más adelante vamos atener que decidir qué hacer con ella.Para la dimensión D2 tenemos que partir esas siete etiquetas en dos grupos. La decisiónque tomamos es esta:

In [ ]:
# Emociones que leemos como hostilidad dirigidaEMOCIONES_VIOLENTAS = ['anger', 'disgust']# Emociones con carga afectiva que no es hostilEMOCIONES_NO_VIOLENTAS = ['joy', 'sadness', 'surprise', 'fear']# 'others' queda afuera de las dos listas, a propósito (ver abajo)

### Por qué este agrupamiento, y por qué es discutible`anger` y `disgust` van juntas porque son las dos emociones que la literatura sobrehostilidad online asocia con la agresión hacia un otro: el enojo moviliza contra alguieny el asco lo deshumaniza. `joy`, `sadness`, `surprise` y `fear` son emocionalmenteintensas pero no hostiles. `fear` es el caso más incómodo del grupo —el miedo aparecemuchas veces junto al discurso xenófobo— pero por sí solo no es una emoción dirigidacontra alguien, así que lo dejamos del lado no violento.`others` queda **excluida de las dos listas**, y no es un detalle. Si la metiéramos en eldenominador, D2 pasaría a medir sobre todo *cuánta emoción marcada hay* en los comentariosen vez de *de qué tipo es la emoción que hay*, que es lo que queremos. Como `others` es laclase mayoritaria, dominaría la dimensión entera. La reportamos aparte, como una medida decobertura: si en una noticia el 95% de los comentarios cae en `others`, su D2 estácalculada sobre muy poquitos casos y hay que desconfiar.Este mapeo es una **decisión de investigación**, no un resultado del modelo. En el ejerciciofinal vas a poder cambiarlo y ver cuánto se mueve el índice.

# Predicción sobre los comentariosPredecimos en lotes, que es bastante más rápido que comentario por comentario. Con GPU,unos pocos minutos para varios miles de comentarios; sin GPU, andá a hacerte un café.

In [ ]:
def predecir_en_batches(analyzer, textos, batch_size=BATCH_SIZE, desc=''):    """Corre el analyzer en lotes y devuelve la lista de resultados."""    salidas = []    for inicio in tqdm(range(0, len(textos), batch_size), desc=desc):        salidas.extend(analyzer.predict(textos[inicio:inicio + batch_size]))    return salidastextos = list(comentarios['text_prep'])res_sent = predecir_en_batches(analyzer_sent, textos, desc='sentimiento')res_emo = predecir_en_batches(analyzer_emo, textos, desc='emoción')

In [ ]:
comentarios['sent_label'] = [r.output for r in res_sent]comentarios['p_pos'] = [r.probas['POS'] for r in res_sent]comentarios['p_neu'] = [r.probas['NEU'] for r in res_sent]comentarios['p_neg'] = [r.probas['NEG'] for r in res_sent]comentarios['emo_label'] = [r.output for r in res_emo]# Score continuo de sentimiento, en [-1, 1]. Es el insumo de D3.comentarios['score_sent'] = comentarios['p_pos'] - comentarios['p_neg']comentarios[[COLS['texto'], 'sent_label', 'score_sent', 'emo_label']].sample(5, random_state=SEED)

El `score_sent` merece una explicación. En vez de quedarnos solo con la etiqueta discreta,construimos un score continuo $s = P(\text{POS}) - P(\text{NEG})$ que vive en $[-1, 1]$.Un comentario furiosamente negativo da algo cercano a $-1$, uno entusiasta cercano a $+1$,y uno genuinamente ambiguo o neutral queda cerca de $0$.Esto conserva información que la etiqueta tira a la basura: un comentario conprobabilidades $(0.45, 0.10, 0.45)$ y otro con $(0.05, 0.90, 0.05)$ se etiquetan los doscomo `NEU` —o casi—, pero el primero es un comentario sobre el que el modelo duda entrelos dos extremos y el segundo es un comentario plácidamente neutro. Para medir polarizaciónesa diferencia importa.

# Descriptivos a nivel comentarioAntes de agregar, miremos qué pinta tienen las clasificaciones crudas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))fig.patch.set_facecolor(FONDO)# Sentimientosc = comentarios['sent_label'].value_counts().reindex(['NEG', 'NEU', 'POS']).fillna(0)axes[0].bar(sc.index, sc.values, color=[ROJO, GRIS, AZUL])axes[0].set_title('Sentimiento')# Emociónec = comentarios['emo_label'].value_counts()colores_emo = [ROJO if e in EMOCIONES_VIOLENTAS               else (AZUL if e in EMOCIONES_NO_VIOLENTAS else GRIS)               for e in ec.index]axes[1].bar(ec.index, ec.values, color=colores_emo)axes[1].set_title('Emoción (rojo = violentas, gris = others)')axes[1].tick_params(axis='x', rotation=45)# Score continuoaxes[2].hist(comentarios['score_sent'], bins=50, color=AZUL, edgecolor=FONDO)axes[2].axvline(0, color=GRIS, linestyle=':')axes[2].set_title('Score de sentimiento (P(POS) - P(NEG))')for ax in axes:    estilo(ax)plt.tight_layout()plt.show()print(f"Proporción de comentarios en 'others': "      f"{(comentarios['emo_label'] == 'others').mean():.1%}")

### Qué muestran estos descriptivosTres cosas que conviene registrar antes de seguir.La primera: el sentimiento en comentarios de noticias está **fuertemente corrido hacia lonegativo**. Esto no es un hallazgo sobre la Argentina, es una regularidad de las seccionesde comentarios en general —comentar tiene un costo, y quien lo paga suele ser el que estámolesto—. Es exactamente el sesgo de selección que mencionamos al muestrear.La segunda: `others` se lleva una porción enorme de los comentarios. El analyzer deemoción es conservador y solo asigna una emoción de Ekman cuando la señal es clara. Poreso D2 se calcula sobre el subconjunto de comentarios con emoción marcada, y por esoreportamos la cobertura.La tercera, y la más importante para lo que viene: el histograma del score continuomuestra la forma de la distribución **agregada de todo el corpus**. Pero el índice no secalcula sobre esta distribución global, sino sobre la de cada noticia por separado. Dosnoticias pueden aportar histogramas completamente distintos y promediarse en algo que separece a este. Esa es justamente la información que queremos recuperar.

# Construcción del índiceAhora sí, las tres dimensiones. Cada una es una función que toma los comentarios de unanoticia y devuelve un número.

## D1 — Ratio entre comentarios positivos y negativosLa forma más directa de mirar el balance es el ratio $n_{pos}/n_{neg}$. Tiene dosproblemas prácticos y uno conceptual.Los prácticos: no está acotado (puede valer 1, 7 o 300) y se rompe cuando no hay ningúncomentario negativo. Los dos se arreglan tomando el logaritmo del ratio suavizado:$$\text{log-ratio} = \log\frac{n_{pos} + 1}{n_{neg} + 1}$$que queda simétrico alrededor de 0 —positivo si predominan los elogios, negativo sipredominan las críticas— y nunca se indefine.El conceptual es más de fondo: **un ratio muy desbalanceado no indica polarización, indicaconsenso**. Una noticia donde el 95% de los comentarios son negativos tiene un ratioextremo, pero ahí no hay ninguna grieta: todos opinan lo mismo. La polarización está en elotro lado, cuando el ratio se acerca a 1 y la sección está partida al medio.Por eso el componente que entra al índice no es el ratio sino su distancia al equilibrio:$$d_1 = 1 - \left|\frac{n_{pos} - n_{neg}}{n_{pos} + n_{neg}}\right| \in [0, 1]$$Vale 1 cuando hay exactamente tantos positivos como negativos y 0 cuando son todos delmismo signo. Reportamos las dos cosas: el log-ratio como descriptivo del tono, y $d_1$como medida de división.

In [ ]:
def disenso_valencia(sent_labels):    """D1: balance entre comentarios positivos y negativos."""    s = pd.Series(sent_labels, dtype='object')    n_pos = int((s == 'POS').sum())    n_neg = int((s == 'NEG').sum())    n_neu = int((s == 'NEU').sum())    log_ratio = float(np.log((n_pos + 1) / (n_neg + 1)))    d1 = 0.0 if n_pos + n_neg == 0 else 1.0 - abs(n_pos - n_neg) / (n_pos + n_neg)    return {'n_pos': n_pos, 'n_neg': n_neg, 'n_neu': n_neu,            'log_ratio_pos_neg': log_ratio, 'd1_disenso': float(d1)}# Chequeo rápido sobre casos que conocemos de antemanoprint(disenso_valencia(['POS'] * 10))                 # unánime -> d1 = 0print(disenso_valencia(['POS'] * 5 + ['NEG'] * 5))    # partido -> d1 = 1

## D2 — Ratio entre emociones violentas y no violentasMisma lógica, sobre las etiquetas de emoción. El ratio crudo es lo que pide la consigna;la versión acotada es la que entra al índice:$$\text{ratio} = \frac{n_{viol} + 1}{n_{noviol} + 1}\qquad\qquadd_2 = \frac{n_{viol}}{n_{viol} + n_{noviol}} \in [0, 1]$$Acá, a diferencia de D1, sí queremos que valores altos signifiquen "más polarizado":una sección donde la mayor parte de la emoción expresada es enojo y asco es más hostilque una donde predominan la alegría o la tristeza. No hay una noción de equilibrio querecuperar.

In [ ]:
def ratio_emociones(emo_labels):    """D2: proporción de emoción hostil sobre el total de emoción marcada."""    s = pd.Series(emo_labels, dtype='object')    n_viol = int(s.isin(EMOCIONES_VIOLENTAS).sum())    n_noviol = int(s.isin(EMOCIONES_NO_VIOLENTAS).sum())    n_others = int((s == 'others').sum())    ratio = float((n_viol + 1) / (n_noviol + 1))    d2 = 0.0 if n_viol + n_noviol == 0 else n_viol / (n_viol + n_noviol)    return {'n_violentas': n_viol, 'n_no_violentas': n_noviol, 'n_others': n_others,            'prop_others': float(n_others / len(s)) if len(s) else np.nan,            'ratio_violentas': ratio, 'd2_violencia': float(d2)}print(ratio_emociones(['anger'] * 5 + ['joy'] * 5))                    # d2 = 0.5print(ratio_emociones(['others'] * 90 + ['anger'] * 5 + ['joy'] * 5))  # others no afecta d2

## D3 — Bimodalidad del sentimientoAcá está el aporte de la tercera dimensión, y conviene entender bien por qué hace falta.Comparemos dos noticias hipotéticas, cada una con 100 comentarios:| | Noticia A | Noticia B ||---|---|---|| Comentarios | 50 `POS` + 50 `NEG` | 100 `NEU` || $n_{pos}/n_{neg}$ | 1 | indefinido (0/0) || $d_1$ | 1 | 0 |En este caso $d_1$ las distingue bien. Pero ahora pensemos en dos noticias que tienen lasdos un $d_1$ alto: una donde los comentarios se reparten entre elogios furiosos e insultos,y otra donde se reparten entre tibios "me parece bien" y tibios "no me convence". Elconteo de etiquetas es el mismo. La conversación no.Lo que falta es mirar la **dispersión del score continuo**, no el conteo de etiquetas:$$d_3 = \frac{\mathrm{Var}(s)}{1 - \bar{s}^2} \in [0, 1]$$El numerador es la varianza de los scores de la noticia. El denominador es la varianza*máxima alcanzable* dada esa media: como $s \in [-1, 1]$, vale$\mathrm{Var}(s) = E[s^2] - \bar{s}^2 \leq 1 - \bar{s}^2$. Dividir por ese máximonormaliza la medida a $[0, 1]$ y la vuelve comparable entre noticias con distinto tonopromedio.$d_3$ vale 1 solamente si toda la masa está en los extremos $\pm 1$ —dos bandos, sinnadie en el medio— y 0 si todos los comentarios coinciden, sin importar en qué valorcoincidan. Es, literalmente, cuánto se parece la sección de comentarios a dos gruposenfrentados en vez de a una nube alrededor de un promedio.

In [ ]:
def bimodalidad(scores):    """D3: dispersión del score de sentimiento, normalizada a [0, 1]."""    s = np.asarray(scores, dtype=float)    s = s[~np.isnan(s)]    if len(s) < 2:        return np.nan    media = float(s.mean())    var = float(s.var(ddof=0))    denom = 1.0 - media ** 2    # media = ±1 implica que todos los scores valen exactamente ±1: unanimidad    # extrema, que no es polarización.    if denom <= 1e-12:        return 0.0    return float(np.clip(var / denom, 0.0, 1.0))print(f"50 POS extremos + 50 NEG extremos -> d3 = {bimodalidad([1.0]*50 + [-1.0]*50):.2f}")print(f"100 comentarios neutrales         -> d3 = {bimodalidad([0.0]*100):.2f}")print(f"100 comentarios todos positivos   -> d3 = {bimodalidad([1.0]*100):.2f}")

### Una alternativa de la literatura: Esteban-Ray$d_3$ es una medida razonable pero armada por nosotros. La literatura de economía políticatiene una familia de índices pensados exactamente para esto, y el más conocido es el de**Esteban y Ray (1994)**:$$ER = K \sum_i \sum_j \pi_i^{1+\alpha}\, \pi_j\, |y_i - y_j|$$donde $\pi_i$ es la proporción del grupo $i$ e $y_i$ su posición. La idea es que lapolarización crece con dos cosas a la vez: la **identificación** (grupos internamentegrandes y homogéneos, capturada por el exponente $1+\alpha$) y la **alienación** (gruposlejanos entre sí, capturada por $|y_i - y_j|$). Un montón de grupitos dispersos no espolarización; dos bloques grandes y opuestos, sí.Lo aplicamos sobre las tres clases discretas ubicadas en $\{-1, 0, +1\}$, con$\alpha = 1.6$, que es el tope del rango que los autores admiten. La constante $K$normaliza a $[0,1]$: el máximo de la suma se alcanza en $\pi = (0.5, 0, 0.5)$ y vale$2^{-\alpha}$, así que $K = 2^{\alpha}$.Lo calculamos en paralelo a $d_3$ para poder comparar las dos operacionalizaciones másabajo. Que dos medidas construidas de forma distinta ordenen las noticias parecido esevidencia de que estamos midiendo algo real y no un artefacto de nuestra fórmula.

In [ ]:
def esteban_ray(props, alpha=1.6, posiciones=(-1.0, 0.0, 1.0)):    """Índice de polarización de Esteban-Ray sobre las clases (NEG, NEU, POS)."""    pi = np.asarray(props, dtype=float)    y = np.asarray(posiciones, dtype=float)    if pi.sum() <= 0:        return np.nan    pi = pi / pi.sum()    bruto = sum(pi[i] ** (1 + alpha) * pi[j] * abs(y[i] - y[j])                for i in range(len(pi)) for j in range(len(pi)))    return float(np.clip(bruto * (2.0 ** alpha), 0.0, 1.0))print(f"(0.5, 0, 0.5) -> ER = {esteban_ray((0.5, 0.0, 0.5)):.2f}")   # máximoprint(f"(0, 1, 0)     -> ER = {esteban_ray((0.0, 1.0, 0.0)):.2f}")   # todos neutralesprint(f"(0.33 c/u)    -> ER = {esteban_ray((1/3, 1/3, 1/3)):.2f}")   # disperso, no polarizado

## El índice compuestoCon las tres dimensiones en $[0,1]$, el índice es su promedio simple:$$IP = \frac{d_1 + d_2 + d_3}{3}$$Que las tres pesen igual **es una decisión, no un resultado**. Estamos afirmando que estardividido, ser hostil y estar bimodal contribuyen lo mismo a "estar polarizado", y no haynada en los datos que lo justifique. Más abajo comparamos contra una ponderación derivadade los datos (componentes principales) para ver cuánto cambia el ordenamiento.

In [ ]:
# Las tres dimensiones que entran al índice compuestoDIMS = ['d1_disenso', 'd2_violencia', 'd3_bimodalidad']def indice_polarizacion(comentarios, col_noticia, min_comentarios=MIN_COMENTARIOS):    """Agrega los comentarios clasificados en un índice por noticia.    Espera un DataFrame con una fila por comentario y las columnas `sent_label`,    `emo_label` y `score_sent`. Devuelve una fila por noticia.    """    filas = []    for noticia_id, grupo in comentarios.groupby(col_noticia, sort=False):        n = len(grupo)        if n < min_comentarios:            continue        fila = {col_noticia: noticia_id, 'n_comentarios': n}        fila.update(disenso_valencia(grupo['sent_label']))        fila.update(ratio_emociones(grupo['emo_label']))        fila['d3_bimodalidad'] = bimodalidad(grupo['score_sent'])        total = fila['n_neg'] + fila['n_neu'] + fila['n_pos']        fila['esteban_ray'] = esteban_ray(            (fila['n_neg'] / total, fila['n_neu'] / total, fila['n_pos'] / total)        ) if total > 0 else np.nan        filas.append(fila)    noticias = pd.DataFrame(filas)    if noticias.empty:        return noticias    noticias['indice_polarizacion'] = noticias[DIMS].mean(axis=1)    return noticias.sort_values('indice_polarizacion', ascending=False).reset_index(drop=True)noticias = indice_polarizacion(comentarios, col_noticia=COLS['noticia'])# Le pegamos los metadatos de cada noticia (título, medio, fecha)meta_cols = [c for c in (COLS['titulo'], COLS['medio'], COLS['fecha']) if c]if meta_cols:    meta = comentarios.groupby(COLS['noticia'])[meta_cols].first().reset_index()    noticias = noticias.merge(meta, on=COLS['noticia'], how='left')print(f'Índice calculado para {len(noticias):,} noticias')noticias[['n_comentarios'] + DIMS + ['esteban_ray', 'indice_polarizacion']].describe().round(3)

# Análisis descriptivoYa tenemos el índice. Ahora la parte que importa: qué nos dice.

## Cómo se distribuye cada dimensión

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))fig.patch.set_facecolor(FONDO)titulos = ['D1 — disenso de valencia', 'D2 — violencia emocional',           'D3 — bimodalidad', 'Índice compuesto']colores = [AZUL, ROJO, AZUL_OSCURO, GRIS]for ax, col, titulo, color in zip(axes, DIMS + ['indice_polarizacion'], titulos, colores):    ax.hist(noticias[col].dropna(), bins=25, color=color, edgecolor=FONDO)    ax.axvline(noticias[col].mean(), color=GRIS, linestyle='--', linewidth=1)    ax.set_title(titulo, fontsize=10)    ax.set_xlim(0, 1)    estilo(ax)plt.tight_layout()plt.show()noticias[DIMS + ['indice_polarizacion']].describe().round(3)

## ¿Las tres dimensiones miden lo mismo?Si estuvieran fuertemente correlacionadas entre sí, tener tres sería redundante: bastaríacon una. Si no lo estuvieran para nada, promediarlas sería sospechoso, porque estaríamossumando cosas que no tienen nada que ver.

In [ ]:
corr = noticias[DIMS + ['esteban_ray', 'indice_polarizacion']].corr()fig, ax = plt.subplots(figsize=(6, 5))fig.patch.set_facecolor(FONDO)sns.heatmap(corr, annot=True, fmt='.2f', cmap=blue_ramp, cbar=False,            linewidths=2, linecolor=FONDO, vmin=-1, vmax=1, ax=ax)ax.set_title('Correlación entre las dimensiones')plt.tight_layout()plt.show()

### Cómo leer esta matrizLo esperable —y conviene verificarlo en la salida, no darlo por sentado— es que $d_1$ y$d_3$ estén bastante correlacionadas, porque las dos salen del analyzer de sentimiento ycapturan variantes de la misma idea de división. $d_2$ debería ser la más independiente:viene de otro modelo y mide otra cosa, hostilidad en vez de desacuerdo.Esa independencia es lo que justifica tener tres dimensiones. Una noticia puede estar muydividida sin ser hostil —un debate cordial— y puede ser hostil sin estar dividida —todosenojados contra el mismo blanco—. Son fenómenos distintos y el índice compuesto los mezclaa propósito; si te interesa uno de los dos en particular, usá la dimensión suelta y no elcompuesto.

## Las noticias más y menos polarizantes

In [ ]:
cols_tabla = [c for c in [COLS['titulo'], COLS['medio']] if c] + \             ['n_comentarios'] + DIMS + ['indice_polarizacion']print('LAS 10 NOTICIAS MÁS POLARIZANTES')display(noticias.head(10)[cols_tabla].round(3))print('\nLAS 10 NOTICIAS MENOS POLARIZANTES')display(noticias.tail(10)[cols_tabla].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))fig.patch.set_facecolor(FONDO)col_etiqueta = COLS['titulo'] if COLS['titulo'] else COLS['noticia']for ax, sub, titulo, color in [    (axes[0], noticias.head(10), 'Más polarizantes', ROJO),    (axes[1], noticias.tail(10), 'Menos polarizantes', AZUL),]:    etiquetas = sub[col_etiqueta].astype(str).str.slice(0, 55)    ax.barh(range(len(sub)), sub['indice_polarizacion'], color=color)    ax.set_yticks(range(len(sub)))    ax.set_yticklabels(etiquetas, fontsize=8)    ax.invert_yaxis()    ax.set_xlim(0, 1)    ax.set_xlabel('Índice de polarización')    ax.set_title(titulo)    estilo(ax)plt.tight_layout()plt.show()

## Polarización por medio

In [ ]:
if COLS['medio']:    orden = (noticias.groupby(COLS['medio'])['indice_polarizacion']             .median().sort_values(ascending=False).index)    fig, axes = plt.subplots(1, 2, figsize=(14, 4))    fig.patch.set_facecolor(FONDO)    sns.boxplot(data=noticias, x=COLS['medio'], y='indice_polarizacion',                order=orden, color=AZUL, ax=axes[0], fliersize=2)    axes[0].set_title('Índice compuesto por medio')    axes[0].tick_params(axis='x', rotation=30)    resumen_medio = (noticias.groupby(COLS['medio'])[DIMS].mean().loc[orden])    resumen_medio.plot(kind='bar', ax=axes[1], rot=30,                       color=[AZUL, ROJO, AZUL_OSCURO])    axes[1].set_title('Dimensiones por medio')    axes[1].legend(frameon=False, fontsize=8)    for ax in axes:        estilo(ax)    plt.tight_layout()    plt.show()    display(noticias.groupby(COLS['medio'])            .agg(n_noticias=('n_comentarios', 'size'),                 comentarios=('n_comentarios', 'sum'),                 indice_medio=('indice_polarizacion', 'mean'))            .round(3).sort_values('indice_medio', ascending=False))else:    print('El dataset no trae columna de medio: salteamos este análisis.')

### Un caveat importante sobre la comparación entre mediosEs tentador leer estas diferencias como "el medio X polariza más que el medio Y". Antesde hacerlo, tres advertencias.Primero, los medios no cubren los mismos temas ni en la misma proporción. Si uno hace máspoliciales y otro más economía, la diferencia entre ellos puede ser enteramente un efectode **composición temática**, no del medio. Para separar una cosa de la otra habría quecomparar dentro de un mismo tema.Segundo, las audiencias son distintas y autoseleccionadas. El índice mide a quienescomentan, que no son ni el público del medio ni la población.Tercero, la cantidad de noticias por medio en nuestra muestra puede ser muy despareja—mirá la columna `n_noticias`—, y con pocas noticias la media de un medio es inestable.

## ¿El volumen de comentarios predice polarización?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))fig.patch.set_facecolor(FONDO)r_vol = noticias[['n_comentarios', 'indice_polarizacion']].corr().iloc[0, 1]axes[0].scatter(noticias['n_comentarios'], noticias['indice_polarizacion'],                color=AZUL, alpha=0.6, s=25)axes[0].set_xscale('log')axes[0].set_xlabel('Comentarios (escala log)')axes[0].set_ylabel('Índice de polarización')axes[0].set_title(f'Índice vs. volumen (r = {r_vol:.2f})')r_er = noticias[['d3_bimodalidad', 'esteban_ray']].corr().iloc[0, 1]axes[1].scatter(noticias['d3_bimodalidad'], noticias['esteban_ray'],                color=AZUL_OSCURO, alpha=0.6, s=25)axes[1].plot([0, 1], [0, 1], color=GRIS, linestyle=':', label='identidad')axes[1].set_xlabel('D3 — bimodalidad (varianza normalizada)')axes[1].set_ylabel('Esteban-Ray')axes[1].set_title(f'Dos operacionalizaciones de lo mismo (r = {r_er:.2f})')axes[1].legend(frameon=False)for ax in axes:    estilo(ax)plt.tight_layout()plt.show()

### Qué explican estos dos gráficosEl de la izquierda contesta si las noticias más comentadas son también las más polarizadas.Una correlación positiva fuerte sería incómoda: sugeriría que el índice está capturandosobre todo el tamaño de la muestra y no una propiedad de la conversación. Una correlacióndébil es la buena noticia.El de la derecha es un chequeo de robustez. $d_3$ y Esteban-Ray se construyen distinto—uno con la varianza del score continuo, el otro con las proporciones de las tres clasesdiscretas y una fórmula que viene de la economía política— pero apuntan al mismo concepto.Si ordenan las noticias parecido, la conclusión no depende de nuestra elección de fórmula.Que los puntos no caigan sobre la diagonal no es un problema: las escalas no tienen porqué coincidir, lo que importa es que la nube suba.

## ¿Cuántos comentarios hacen falta? El umbral, justificadoFijamos `MIN_COMENTARIOS = 30` al principio sin explicar por qué. Vamos a justificarloahora, con bootstrap: para cada noticia remuestreamos sus comentarios con reposición 200veces, recalculamos el índice cada vez, y medimos el ancho del intervalo de confianza del95%. Si el índice de una noticia se mueve muchísimo al remuestrear sus propios comentarios,es que ese número no es confiable.

In [ ]:
def bootstrap_indice(grupo, n_rep=200):    """Ancho del IC 95% del índice compuesto, remuestreando los comentarios."""    n = len(grupo)    if n < 2:        return np.nan    sent = grupo['sent_label'].to_numpy()    emo = grupo['emo_label'].to_numpy()    score = grupo['score_sent'].to_numpy()    reps = []    for _ in range(n_rep):        idx = rng.integers(0, n, size=n)        reps.append(np.nanmean([            disenso_valencia(sent[idx])['d1_disenso'],            ratio_emociones(emo[idx])['d2_violencia'],            bimodalidad(score[idx]),        ]))    return float(np.nanpercentile(reps, 97.5) - np.nanpercentile(reps, 2.5))anchos = pd.DataFrame([    {COLS['noticia']: nid, 'ancho_ic': bootstrap_indice(g)}    for nid, g in tqdm(comentarios.groupby(COLS['noticia'], sort=False), desc='bootstrap')])anchos = anchos.merge(noticias[[COLS['noticia'], 'n_comentarios']], on=COLS['noticia'])fig, ax = plt.subplots(figsize=(7, 4.5))fig.patch.set_facecolor(FONDO)ax.scatter(anchos['n_comentarios'], anchos['ancho_ic'], color=AZUL, alpha=0.6, s=25)ax.axvline(MIN_COMENTARIOS, color=ROJO, linestyle='--',           label=f'MIN_COMENTARIOS = {MIN_COMENTARIOS}')ax.set_xscale('log')ax.set_xlabel('Comentarios en la noticia (escala log)')ax.set_ylabel('Ancho del IC 95% del índice')ax.set_title('Cuánto se mueve el índice al remuestrear los comentarios')ax.legend(frameon=False)estilo(ax)plt.tight_layout()plt.show()print(f"Ancho medio del IC 95%: {anchos['ancho_ic'].mean():.3f}")print(f"Ancho medio en noticias con menos de 50 comentarios:  "      f"{anchos.loc[anchos['n_comentarios'] < 50, 'ancho_ic'].mean():.3f}")print(f"Ancho medio en noticias con 100 o más comentarios:    "      f"{anchos.loc[anchos['n_comentarios'] >= 100, 'ancho_ic'].mean():.3f}")

### Qué explica este gráficoLa nube baja de izquierda a derecha: cuantos más comentarios tiene una noticia, másestable es su índice. Esto es lo esperable —es la misma lógica por la que una encuesta de1000 casos tiene menos margen de error que una de 100— pero verlo tiene una consecuenciapráctica concreta.Si el ancho del intervalo para una noticia de 30 comentarios es, digamos, 0,15, entonces**dos noticias que difieren en menos de 0,15 puntos de índice no son distinguibles**. Esodescalifica cualquier lectura del tipo "la noticia número 3 del ranking polariza más quela número 5". El índice sirve para comparar extremos y para trabajar con promedios degrupos, no para ordenar noticias una por una en el medio de la distribución.Ahí está la justificación del umbral: 30 es el punto donde el intervalo se vuelve lobastante angosto para las comparaciones gruesas que nos interesan. Es un compromiso entreprecisión y cuántas noticias nos quedan, no una verdad estadística.

## Una ponderación alternativaPromediar las tres dimensiones con peso igual fue una decisión nuestra. Una alternativa esdejar que los datos sugieran los pesos, vía componentes principales: el primer componentees la combinación lineal de las tres dimensiones que más varianza explica.

In [ ]:
X = noticias[DIMS].dropna().valuesdesvios = X.std(axis=0)desvios[desvios == 0] = 1.0          # una dimensión constante no aporta varianzaX_std = (X - X.mean(axis=0)) / desviospca = PCA(n_components=len(DIMS)).fit(X_std)pesos = pca.components_[0]if pesos.sum() < 0:          # el signo del componente es arbitrario    pesos = -pesosnoticias_pca = noticias[DIMS].dropna().copy()noticias_pca['ip_pca'] = X_std @ pesosnoticias_pca['ip_simple'] = noticias.loc[noticias_pca.index, 'indice_polarizacion']r_pca = noticias_pca[['ip_simple', 'ip_pca']].corr().iloc[0, 1]rho_pca = noticias_pca[['ip_simple', 'ip_pca']].corr(method='spearman').iloc[0, 1]print(f'Varianza explicada por el primer componente: {pca.explained_variance_ratio_[0]:.1%}')print(f'Pesos del primer componente:')for dim, peso in zip(DIMS, pesos):    print(f'  {dim:<18} {peso:+.3f}')print(f'\nCorrelación con el índice de pesos iguales: r = {r_pca:.3f}')print(f'Correlación de rangos (Spearman):            rho = {rho_pca:.3f}')fig, ax = plt.subplots(figsize=(5.5, 5))fig.patch.set_facecolor(FONDO)ax.scatter(noticias_pca['ip_simple'], noticias_pca['ip_pca'], color=AZUL, alpha=0.6, s=25)ax.set_xlabel('Índice con pesos iguales')ax.set_ylabel('Índice con pesos del primer componente')ax.set_title(f'Las dos ponderaciones (rho = {rho_pca:.2f})')estilo(ax)plt.tight_layout()plt.show()

### Qué explica esta comparaciónSi la correlación de rangos es alta —digamos por encima de 0,9— entonces la elección deponderación **no cambia las conclusiones**, y eso es tranquilizador: significa que elranking de noticias no es un artefacto de haber elegido pesos iguales.Vale aclarar que PCA no es "la ponderación objetiva". PCA maximiza varianza explicada, queno es lo mismo que maximizar validez: le da más peso a las dimensiones que más varían entrenoticias, no a las que mejor capturan el concepto de polarización. Es una alternativa útilpara chequear robustez, no un árbitro.

## Serie temporal

In [ ]:
if COLS['fecha']:    serie = (noticias.assign(_fecha=pd.to_datetime(noticias[COLS['fecha']], errors='coerce'))             .dropna(subset=['_fecha'])             .set_index('_fecha')['indice_polarizacion']             .resample('W').agg(['mean', 'size']))    serie = serie[serie['size'] >= 3]      # semanas con muy pocas noticias son ruido    if len(serie) > 1:        fig, ax = plt.subplots(figsize=(11, 3.8))        fig.patch.set_facecolor(FONDO)        ax.plot(serie.index, serie['mean'], color=AZUL, marker='o', markersize=4)        ax.axhline(noticias['indice_polarizacion'].mean(), color=GRIS, linestyle='--',                   linewidth=1, label='promedio general')        ax.set_ylabel('Índice promedio')        ax.set_title('Índice de polarización promedio por semana')        ax.legend(frameon=False)        estilo(ax)        plt.tight_layout()        plt.show()    else:        print('No hay suficientes semanas con datos para una serie temporal.')else:    print('El dataset no trae columna de fecha: salteamos este análisis.')

## Lectura cualitativaEste es el paso que más se saltea y el más importante. Todo lo anterior son números quesalieron de un modelo que se puede estar equivocando de forma sistemática. La única manerade saber si el índice mide lo que creemos que mide es **leer los comentarios**.Si la noticia con índice más alto no parece polarizada al leerla, el índice está mal —o elmodelo está clasificando mal, que para el caso es lo mismo—.

In [ ]:
def mostrar_noticia(fila, n_ejemplos=8):    """Imprime el índice de una noticia y una muestra de sus comentarios."""    sub = comentarios[comentarios[COLS['noticia']] == fila[COLS['noticia']]]    if COLS['titulo']:        print(f'NOTICIA: {fila[COLS["titulo"]]}')    if COLS['medio']:        print(f'MEDIO: {fila[COLS["medio"]]}')    print(f'n = {fila["n_comentarios"]} comentarios | '          f'IP = {fila["indice_polarizacion"]:.3f} '          f'(d1={fila["d1_disenso"]:.2f}, d2={fila["d2_violencia"]:.2f}, '          f'd3={fila["d3_bimodalidad"]:.2f})')    print('-' * 100)    # Mostramos los más positivos y los más negativos, que es donde se ve la grieta    ordenados = sub.sort_values('score_sent')    seleccion = pd.concat([ordenados.head(n_ejemplos // 2),                           ordenados.tail(n_ejemplos // 2)])    for _, c in seleccion.iterrows():        print(f'[{c["sent_label"]:>3} {c["score_sent"]:+.2f} | {c["emo_label"]:<8}] '              f'{str(c[COLS["texto"]])[:160]}')    print()print('=' * 100)print('LA NOTICIA MÁS POLARIZADA')print('=' * 100)mostrar_noticia(noticias.iloc[0])print('=' * 100)print('LA NOTICIA MENOS POLARIZADA')print('=' * 100)mostrar_noticia(noticias.iloc[-1])

### Qué mirar en esta salidaTres preguntas concretas para hacerle a estos comentarios.**¿La noticia más polarizada tiene efectivamente dos bandos?** Si al leer los comentariosmás positivos y los más negativos ves dos posiciones enfrentadas sobre el mismo asunto,el índice está funcionando. Si en cambio los "positivos" son ironías o sarcasmo malclasificado, tenés un problema: el modelo no detecta ironía y en comentarios políticos laironía es moneda corriente.**¿Los `score_sent` extremos se corresponden con lo que leés?** Un comentario con score$-0.95$ debería resultarte claramente hostil. Si no, el modelo está sobreconfiado.**¿La noticia menos polarizada es realmente tranquila, o simplemente no habla de nada?**A veces el índice bajo no indica consenso sino que los comentarios son spam, saludos oestán fuera de tema. Eso no es una conversación despolarizada, es ausencia de conversación,y son dos cosas distintas que el índice no distingue.

# Conclusiones**Sobre el procedimiento.** El movimiento central de esta notebook fue pasar de clasificartextos a medir una propiedad de un colectivo. `pysentimiento` hizo la parte fácil: dosllamadas a `create_analyzer` y un `.predict()`. Todo el trabajo intelectual estuvo en lasdecisiones de agregación —qué emociones cuentan como violentas, qué hacer con `others`,cuántos comentarios hacen falta, cómo ponderar— y ninguna de esas la resuelve el modelo.Cuando en un paper leas "medimos polarización con NLP", las decisiones que importan sonestas, no cuál fue el transformer.**Sobre las tres dimensiones.** No miden lo mismo y eso es deliberado. D1 mide división,D2 mide hostilidad, D3 mide concentración en los extremos. Una noticia puede tener altoD2 y bajo D1 —todos enojados contra el mismo blanco, que es indignación colectiva y nopolarización— o alto D1 y bajo D2 —un desacuerdo civilizado—. El índice compuesto losmezcla; si tu pregunta de investigación es sobre uno de esos fenómenos en particular,usá la dimensión suelta.**Sobre qué NO mide este índice.** Tres límites que hay que declarar antes de que losdeclare quien te revise el trabajo:- **Sentimiento negativo no es polarización.** Una sección donde todos putean al mismo  actor es negativa y unánime. El índice de dos dimensiones que pedía la consigna original  la habría marcado como polarizada; por eso agregamos D3.- **Quienes comentan no son la población.** Comentar tiene un costo y lo paga sobre todo  quien tiene una opinión fuerte. Todo lo que medimos es sobre esa población  autoseleccionada, no sobre los lectores ni sobre la sociedad.- **El modelo tiene sus propios sesgos.** RoBERTuito fue fine-tuneado sobre tuits anotados  por un grupo específico de personas con una definición específica de cada emoción. Ironía,  sarcasmo y jerga regional son sus puntos débiles, y los comentarios políticos argentinos  están hechos de eso. Cada error de clasificación se propaga al índice.**Sobre la precisión.** El bootstrap mostró que el índice de una noticia individual tieneun intervalo de confianza nada despreciable. Sirve para comparar extremos y promedios degrupos, no para rankear noticias una por una.

# Ejercicio1. **Sensibilidad del mapeo de emociones.** Movete `fear` de `EMOCIONES_NO_VIOLENTAS` a   `EMOCIONES_VIOLENTAS` y volvé a correr el índice. ¿Cuánto cambia el ranking de noticias   (compará con la correlación de Spearman entre el ranking viejo y el nuevo)? Si cambia   mucho, la conclusión depende de una decisión tuya que no está justificada por los datos;   si cambia poco, el resultado es robusto. Probá también incluir `others` en el   denominador de D2 y mirá qué le pasa a la dimensión.2. **Una cuarta dimensión: discurso de odio.** Agregá   `create_analyzer(task="hate_speech", lang="es")` —el de la notebook anterior— y calculá   por noticia la proporción de comentarios con la etiqueta `hateful`. ¿Correlaciona con D2?   Si correlacionaran casi perfecto, ¿tendría sentido tener las dos dimensiones?3. **Validación contra lectura humana.** Elegí 20 noticias al azar de la muestra, leelas sin   mirar el índice y ordenalas a mano de más a menos polarizada. Después comparalas con el   ranking del índice (Spearman de nuevo). Esta es la validación que de verdad importa y   casi nunca se hace. Si te da bajo, la pregunta interesante no es "¿el índice está mal?"   sino "¿en qué casos concretos discrepamos y por qué?".4. **De la noticia al tema.** Si el dataset trae una columna de sección o tema, agregá el   índice a ese nivel en vez de por noticia. ¿Qué temas polarizan más? Ojo con el punto que   discutimos sobre la comparación entre medios: el mismo problema de composición aparece   acá al revés.